# DCC 2026 대학부 예선 — EDA

이 노트북은 DCC 2026 대학부 예선용 탐색적 데이터 분석(EDA) 초안입니다.

## 반영한 핵심 규칙
- Training에는 서울 데이터만 사용
- Validation 데이터는 학습에 사용하지 않음
- Mission 1: 성별 분류, Accuracy
- Mission 2: 발화 음성으로 신고자/119대원 분류, text와 발화 순서 사용 금지
- Mission 3: 전사 text만 사용해 9개 symptom multi-label 분류, Macro F1
- Mission 3의 9개 외 symptom은 label만 제거하고 sample은 유지

> 이 노트북의 통합 DataFrame은 EDA용입니다. Mission별 허용 입력 규칙과 모델 feature는 반드시 분리하세요.

## 1. 라이브러리

In [ ]:
from pathlib import Path
import json
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## 2. 데이터 경로 설정

`DATA_ROOT`만 실제 데이터 경로로 수정하세요.

In [ ]:
DATA_ROOT = Path("../data")  # TODO: 실제 경로로 수정

assert DATA_ROOT.exists(), f"DATA_ROOT가 존재하지 않습니다: {DATA_ROOT.resolve()}"
print("DATA_ROOT =", DATA_ROOT.resolve())

## 3. 폴더 구조 확인

In [ ]:
max_depth = 3

for p in sorted(DATA_ROOT.rglob("*")):
    if not p.is_dir():
        continue
    depth = len(p.relative_to(DATA_ROOT).parts)
    if depth <= max_depth:
        print("  " * (depth - 1) + f"[DIR] {p.name}")

## 4. WAV / JSON 탐색

In [ ]:
all_wav_files = sorted(DATA_ROOT.rglob("*.wav"))
all_json_files = sorted(DATA_ROOT.rglob("*.json"))

print(f"전체 WAV 파일 수 : {len(all_wav_files):,}")
print(f"전체 JSON 파일 수: {len(all_json_files):,}")

print("\nWAV 예시")
for p in all_wav_files[:5]:
    print("-", p)

print("\nJSON 예시")
for p in all_json_files[:5]:
    print("-", p)

## 5. 서울 Training 후보 자동 탐색

경로 문자열에 `Training/Train` 과 `서울/Seoul`이 포함된 파일을 우선 찾습니다.
자동 탐색이 실패하면 다음 셀에서 직접 경로를 지정하세요.

In [ ]:
TRAIN_KEYWORDS = ["training", "train"]
SEOUL_KEYWORDS = ["서울", "seoul"]

def has_any_keyword(path, keywords):
    s = str(path).lower()
    return any(k.lower() in s for k in keywords)

train_json_candidates = [p for p in all_json_files if has_any_keyword(p, TRAIN_KEYWORDS)]
train_wav_candidates = [p for p in all_wav_files if has_any_keyword(p, TRAIN_KEYWORDS)]

seoul_train_json_candidates = [p for p in train_json_candidates if has_any_keyword(p, SEOUL_KEYWORDS)]
seoul_train_wav_candidates = [p for p in train_wav_candidates if has_any_keyword(p, SEOUL_KEYWORDS)]

print(f"Training JSON 후보      : {len(train_json_candidates):,}")
print(f"서울 Training JSON 후보 : {len(seoul_train_json_candidates):,}")
print(f"Training WAV 후보       : {len(train_wav_candidates):,}")
print(f"서울 Training WAV 후보  : {len(seoul_train_wav_candidates):,}")

for p in seoul_train_json_candidates[:10]:
    print("-", p)

In [ ]:
train_json_files = seoul_train_json_candidates
train_wav_files = seoul_train_wav_candidates

# 자동 탐색이 실패하면 아래처럼 직접 지정하세요.
# TRAIN_JSON_ROOT = DATA_ROOT / "..."
# TRAIN_WAV_ROOT = DATA_ROOT / "..."
# train_json_files = sorted(TRAIN_JSON_ROOT.rglob("*.json"))
# train_wav_files = sorted(TRAIN_WAV_ROOT.rglob("*.wav"))

print(f"EDA에 사용할 서울 Training JSON: {len(train_json_files):,}")
print(f"EDA에 사용할 서울 Training WAV : {len(train_wav_files):,}")

assert len(train_json_files) > 0, "서울 Training JSON 경로를 찾지 못했습니다. 직접 지정하세요."

## 6. JSON 실제 구조 확인

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

sample_json = load_json(train_json_files[0])

print("샘플:", train_json_files[0])
print("\nTop-level keys:")
print(list(sample_json.keys()))

print("\nTop-level value type:")
for k, v in sample_json.items():
    print(f"{k:20s} -> {type(v).__name__}")

In [ ]:
from pprint import pprint

UTTERANCE_KEYS = ["utterences", "utterances"]

def get_utterances(data):
    for key in UTTERANCE_KEYS:
        if key in data and isinstance(data[key], list):
            return data[key]
    return []

sample_utterances = get_utterances(sample_json)

print("발화 수:", len(sample_utterances))
pprint(sample_utterances[:3])

## 7. WAV ↔ JSON stem 대응 확인

In [ ]:
json_stems = {p.stem for p in train_json_files}
wav_stems = {p.stem for p in train_wav_files}

matched = json_stems & wav_stems
only_json = sorted(json_stems - wav_stems)
only_wav = sorted(wav_stems - json_stems)

print(f"stem 기준 매칭 수 : {len(matched):,}")
print(f"JSON만 존재       : {len(only_json):,}")
print(f"WAV만 존재        : {len(only_wav):,}")
print("JSON만 존재 예시:", only_json[:10])
print("WAV만 존재 예시 :", only_wav[:10])

## 8. 파싱 보조 함수

In [ ]:
TARGET_SYMPTOMS = [
    "고열", "구토", "두통", "복통", "어지러움",
    "열상", "오심", "전신쇠약", "호흡곤란"
]

def normalize_symptom_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        result = []
        for x in value:
            if isinstance(x, str):
                x = x.strip()
                if x:
                    result.append(x)
            elif isinstance(x, dict):
                for key in ("name", "label", "symptom", "value"):
                    if isinstance(x.get(key), str):
                        s = x[key].strip()
                        if s:
                            result.append(s)
                        break
        return result

    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        for sep in [",", "|", ";"]:
            if sep in text:
                return [x.strip() for x in text.split(sep) if x.strip()]
        return [text]

    return []

def filter_target_symptoms(value):
    return [s for s in normalize_symptom_list(value) if s in TARGET_SYMPTOMS]

def safe_numeric(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

## 9. 통화 단위 DataFrame

Mission 1/3 EDA의 기본 테이블입니다.

In [ ]:
call_rows = []
failed_json = []

for json_path in train_json_files:
    try:
        data = load_json(json_path)
        utterances = get_utterances(data)

        texts = []
        for u in utterances:
            if isinstance(u, dict):
                t = u.get("text")
                if isinstance(t, str) and t.strip():
                    texts.append(t.strip())

        full_text = " ".join(texts)
        raw_symptom = data.get("symptom")

        call_rows.append({
            "json_file": json_path.name,
            "json_path": str(json_path),
            "record_id": data.get("recordId"),
            "audio_path_in_json": data.get("audioPath"),
            "gender": data.get("gender"),
            "raw_symptoms": normalize_symptom_list(raw_symptom),
            "target_symptoms": filter_target_symptoms(raw_symptom),
            "num_utterances": len(utterances),
            "text": full_text,
            "text_length_chars": len(full_text),
            "startAt": safe_numeric(data.get("startAt")),
            "endAt": safe_numeric(data.get("endAt")),
        })
    except Exception as e:
        failed_json.append((str(json_path), repr(e)))

call_df = pd.DataFrame(call_rows)

print(f"정상 파싱 JSON: {len(call_df):,}")
print(f"실패 JSON     : {len(failed_json):,}")
display(call_df.head())

## 10. 기본 무결성 점검

In [ ]:
summary = pd.Series({
    "전체 통화 수": len(call_df),
    "record_id 결측": call_df["record_id"].isna().sum(),
    "record_id 중복": call_df["record_id"].duplicated().sum(),
    "gender 결측": call_df["gender"].isna().sum(),
    "text 비어있음": (call_df["text_length_chars"] == 0).sum(),
    "발화 0개": (call_df["num_utterances"] == 0).sum(),
})

display(summary.to_frame("count"))

if failed_json:
    print("\n파싱 실패 예시")
    for item in failed_json[:10]:
        print(item)

# Mission 1 EDA — 성별 분류

평가 지표는 Accuracy입니다.

In [ ]:
gender_counts = call_df["gender"].value_counts(dropna=False)
gender_ratio = call_df["gender"].value_counts(normalize=True, dropna=False)

display(gender_counts.to_frame("count"))
display(gender_ratio.to_frame("ratio"))

In [ ]:
gender_counts.dropna().plot(kind="bar")
plt.title("Mission 1 - Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
display(
    call_df.groupby("gender")[["num_utterances", "text_length_chars"]]
    .describe()
    .round(2)
)

# Mission 2 EDA — 신고자 / 119대원 분류

- 발화 구간: `startAt` ~ `endAt`
- target: `speaker`
- **text와 발화 순서는 모델 feature로 사용 금지**

In [ ]:
utterance_rows = []

for json_path in train_json_files:
    data = load_json(json_path)
    utterances = get_utterances(data)

    for idx, u in enumerate(utterances):
        if not isinstance(u, dict):
            continue

        utterance_rows.append({
            "json_file": json_path.name,
            "record_id": data.get("recordId"),
            "utterance_index": idx,   # EDA용. 모델 feature로 사용 금지
            "startAt": safe_numeric(u.get("startAt")),
            "endAt": safe_numeric(u.get("endAt")),
            "speaker": u.get("speaker"),
            "text": u.get("text", ""),  # EDA용. Mission 2 모델에는 사용 금지
        })

utterance_df = pd.DataFrame(utterance_rows)

print("발화 sample 수:", f"{len(utterance_df):,}")
display(utterance_df.head())

In [ ]:
display(utterance_df["speaker"].value_counts(dropna=False).to_frame("count"))
display(utterance_df["speaker"].value_counts(normalize=True, dropna=False).to_frame("ratio"))

utterance_df["speaker"].value_counts().sort_index().plot(kind="bar")
plt.title("Mission 2 - Speaker Distribution")
plt.xlabel("Speaker (0/1)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 11. startAt / endAt 단위 확인

먼저 실제 값을 보고 초(sec)인지 밀리초(ms)인지 확인한 뒤 `TIME_SCALE`을 설정하세요.

In [ ]:
display(utterance_df[["startAt", "endAt", "speaker"]].head(20))

raw_delta = utterance_df["endAt"] - utterance_df["startAt"]
display(raw_delta.describe())

In [ ]:
TIME_SCALE = 1.0  # TODO: sec이면 1.0, ms이면 1000.0

utterance_df["duration_sec"] = (
    utterance_df["endAt"] - utterance_df["startAt"]
) / TIME_SCALE

display(utterance_df["duration_sec"].describe())
display(utterance_df.groupby("speaker")["duration_sec"].describe().round(3))

In [ ]:
short_stats = pd.Series({
    "< 0.5s": (utterance_df["duration_sec"] < 0.5).mean(),
    "< 1.0s": (utterance_df["duration_sec"] < 1.0).mean(),
    "< 2.0s": (utterance_df["duration_sec"] < 2.0).mean(),
}, name="ratio")

display(short_stats.to_frame())

In [ ]:
valid_duration = utterance_df["duration_sec"].dropna()
valid_duration = valid_duration[valid_duration >= 0]

if len(valid_duration) > 0:
    upper = valid_duration.quantile(0.99)
    plt.hist(valid_duration[valid_duration <= upper], bins=50)
    plt.title("Mission 2 - Utterance Duration (<= 99th percentile)")
    plt.xlabel("Duration (sec)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# Mission 3 EDA — 환자 증상 Multi-label 분류

평가 지표는 9개 symptom별 F1-score의 Macro 평균입니다.

In [ ]:
symptom_counter = Counter()

for symptoms in call_df["target_symptoms"]:
    symptom_counter.update(symptoms)

symptom_counts = pd.Series(
    {s: symptom_counter[s] for s in TARGET_SYMPTOMS}
).sort_values(ascending=False)

display(symptom_counts.to_frame("count"))

In [ ]:
symptom_counts.sort_values().plot(kind="barh")
plt.title("Mission 3 - Target Symptom Distribution")
plt.xlabel("Count")
plt.ylabel("Symptom")
plt.tight_layout()
plt.show()

## 12. 9개 이외 symptom 확인

이 label들은 평가 대상에서 제외하지만 sample 자체는 유지합니다.

In [ ]:
all_symptom_counter = Counter()
for symptoms in call_df["raw_symptoms"]:
    all_symptom_counter.update(symptoms)

non_target = {
    k: v for k, v in all_symptom_counter.items()
    if k not in TARGET_SYMPTOMS
}

non_target_series = pd.Series(non_target).sort_values(ascending=False)
print("Target 이외 symptom 종류 수:", len(non_target_series))
display(non_target_series.head(30).to_frame("count"))

## 13. Sample당 target symptom 개수

In [ ]:
call_df["symptom_count"] = call_df["target_symptoms"].apply(len)
symptom_count_dist = call_df["symptom_count"].value_counts().sort_index()

display(symptom_count_dist.to_frame("count"))
print("target symptom 0개 sample:", int((call_df["symptom_count"] == 0).sum()))

In [ ]:
symptom_count_dist.plot(kind="bar")
plt.title("Mission 3 - Number of Target Symptoms per Conversation")
plt.xlabel("Number of symptoms")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 14. Symptom co-occurrence

In [ ]:
pair_counter = Counter()

for symptoms in call_df["target_symptoms"]:
    unique_symptoms = sorted(set(symptoms))
    for pair in combinations(unique_symptoms, 2):
        pair_counter[pair] += 1

cooccurrence_df = pd.DataFrame([
    {"symptom_1": a, "symptom_2": b, "count": count}
    for (a, b), count in pair_counter.items()
]).sort_values("count", ascending=False)

display(cooccurrence_df.head(20))

## 15. Mission 3 텍스트 길이

In [ ]:
display(call_df["text_length_chars"].describe())

display(
    call_df["text_length_chars"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("text_length_chars")
)

In [ ]:
text_lengths = call_df["text_length_chars"]

if len(text_lengths) > 0:
    upper = text_lengths.quantile(0.99)
    plt.hist(text_lengths[text_lengths <= upper], bins=50)
    plt.title("Mission 3 - Conversation Text Length (<= 99th percentile)")
    plt.xlabel("Number of characters")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

## 16. EDA 요약

In [ ]:
eda_summary = {
    "서울 Training 통화 수": len(call_df),
    "서울 Training 발화 수": len(utterance_df),
    "gender 종류 수": call_df["gender"].nunique(dropna=True),
    "Mission3 target 0개 sample 수": int((call_df["symptom_count"] == 0).sum()),
    "Mission3 가장 많은 symptom": symptom_counts.index[0] if len(symptom_counts) else None,
    "Mission3 가장 적은 symptom": symptom_counts.index[-1] if len(symptom_counts) else None,
    "텍스트 길이 중앙값(chars)": float(call_df["text_length_chars"].median()),
    "텍스트 길이 95% 분위(chars)": float(call_df["text_length_chars"].quantile(0.95)),
}

display(pd.Series(eda_summary).to_frame("value"))

# 17. EDA Findings — 실행 후 직접 채우기

## Common
- 서울 Training 통화 수:
- WAV / JSON 매칭 상태:
- 파싱 실패 파일:
- `startAt/endAt` 단위:
- 결측 / 중복:

## Mission 1
- 성별 분포:
- class imbalance 여부:
- shortcut/bias 가능성:
- 다음 baseline에서 주의할 점:

## Mission 2
- speaker 0/1 비율:
- 평균 / 중앙 발화 길이:
- 0.5초 미만 비율:
- 1초 미만 비율:
- 비정상적인 구간:
- padding / crop 관련 고려사항:

## Mission 3
- 가장 많은 symptom:
- 가장 적은 symptom:
- class imbalance:
- target symptom 0개 sample:
- 자주 같이 등장하는 symptom pair:
- text 길이 95% 분위:
- 다음 baseline에서 고려할 점:

## 다음 단계
1. Mission 1: MFCC + 간단한 classifier
2. Mission 2: 발화 crop + MFCC + 간단한 classifier
3. Mission 3: TF-IDF + Logistic Regression multi-label